In [1]:
import duckdb

In [2]:
con = duckdb.connect("../Data/Processed/database.duckdb")

In [21]:
start_date = con.execute("SELECT MIN(Start_date) AS Date FROM Silver;").df()["Date"].astype("string")[0]
end_date = con.execute("SELECT MAX(End_date) AS Date FROM Silver;").df()["Date"].astype("string")[0]

In [22]:
import openmeteo_requests
import duckdb
import pandas as pd
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "bounding_box": "38.7916,-77.1197,38.9959,-76.9094",
    "models": "icon_global",  # or another single-domain model — best_match won't work
    "start_date": start_date,
    "end_date": end_date,
    "daily": ["temperature_2m_max", "apparent_temperature_max", "rain_sum", "showers_sum", "snowfall_sum", "wind_speed_10m_max"],
}
responses = openmeteo.weather_api(url, params = params)

for response in responses:
	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
	daily_apparent_temperature_max = daily.Variables(1).ValuesAsNumpy()
	daily_rain_sum = daily.Variables(2).ValuesAsNumpy()
	daily_showers_sum = daily.Variables(3).ValuesAsNumpy()
	daily_snowfall_sum = daily.Variables(4).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
	
	daily_data = {
		"date": pd.date_range(
			start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = daily.Interval()),
			inclusive = "left"
		)
	}
	
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["apparent_temperature_max"] = daily_apparent_temperature_max
	daily_data["rain_sum"] = daily_rain_sum
	daily_data["showers_sum"] = daily_showers_sum
	daily_data["snowfall_sum"] = daily_snowfall_sum
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	
	daily_dataframe = pd.DataFrame(data = daily_data)
	print("\nDaily data\n", daily_dataframe)
	


Coordinates: 38.75°N -77.125°E
Elevation: nan m asl
Timezone difference to GMT+0: 0s

Daily data
                         date  temperature_2m_max  apparent_temperature_max  \
0  2020-01-01 00:00:00+00:00                 NaN                       NaN   
1  2020-01-02 00:00:00+00:00                 NaN                       NaN   
2  2020-01-03 00:00:00+00:00                 NaN                       NaN   
3  2020-01-04 00:00:00+00:00                 NaN                       NaN   
4  2020-01-05 00:00:00+00:00                 NaN                       NaN   
5  2020-01-06 00:00:00+00:00                 NaN                       NaN   
6  2020-01-07 00:00:00+00:00                 NaN                       NaN   
7  2020-01-08 00:00:00+00:00                 NaN                       NaN   
8  2020-01-09 00:00:00+00:00                 NaN                       NaN   
9  2020-01-10 00:00:00+00:00                 NaN                       NaN   
10 2020-01-11 00:00:00+00:00               

In [ ]:
daily_dataframe["date"] = daily_dataframe["date"].dt.date

In [ ]:
daily_dataframe.columns = ["Date", "Temp_max", "Apparent_temp_max", "Rain", "Shower", "Snow", "Wind_speed_max"]

In [ ]:
daily_dataframe

,Date,Temp_max,Apparent_temp_max,Rain,Shower,Snow,Wind_speed_max
0,2026-07-10,30.500000,35.091816,1.3,0.1,0.0,9.360000
1,2026-07-11,31.400000,35.141941,0.0,0.2,0.0,9.726664
2,2026-07-12,28.799999,31.996994,0.0,2.3,0.0,11.631956
3,2026-07-13,30.250000,32.411770,0.0,0.0,0.0,10.990322
4,2026-07-14,33.099998,37.000824,0.0,0.0,0.0,9.000000
5,2026-07-15,36.750000,41.046959,0.0,0.0,0.0,11.966954
6,2026-07-16,36.400002,40.984795,0.0,0.0,0.0,11.879999
7,2026-07-17,34.000000,38.204281,0.0,0.0,0.0,6.162207
8,2026-07-18,32.400002,37.531460,0.0,5.4,0.0,17.786331
9,2026-07-19,30.900000,34.017036,0.0,0.0,0.0,11.726277


In [ ]:
con.execute("CREATE TABLE DIM_WEATHER AS SELECT * FROM daily_dataframe").df()

,Count
0,15
